# TUẦN 1 · N7 — ĐO MÔI TRƯỜNG THI THẬT

**Chạy notebook này 2 lần: một lần trên Google Colab, một lần trên Kaggle.**

Mục đích: thay mọi phỏng đoán bằng **số đo thật**. Bạn đang quen H100 80GB
(`cv/PLAN.md`: *"~68GB trống"*). Vòng miền bạn sẽ có **T4 16GB**. Chênh 15–25 lần.

Cuối notebook sẽ in ra một khối Markdown — **copy vào `tuan01/env_report.md`**.

> ⚠️ Kaggle: bật GPU ở *Settings → Accelerator*. Cần xác minh số điện thoại.
> ⚠️ Colab: *Runtime → Change runtime type → T4 GPU*.

## 0 · Phần cứng được cấp

In [ ]:
import subprocess, os, sys, platform, json, time
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
import torch
print("torch      :", torch.__version__, "| CUDA:", torch.version.cuda)
print("GPU        :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("VRAM       :", round(p.total_memory/2**30, 1), "GB")
    print("compute cap:", f"{p.major}.{p.minor}", "| bf16:", torch.cuda.is_bf16_supported())
print("CPU cores  :", os.cpu_count())
print("RAM        :", round(os.sysconf('SC_PAGE_SIZE')*os.sysconf('SC_PHYS_PAGES')/2**30, 1), "GB")
print("Disk free  :", subprocess.run(["df","-h","/"],capture_output=True,text=True).stdout.splitlines()[1])

**Ghi lại:** `bf16` = False trên T4 → bắt buộc dùng **fp16 + GradScaler**, không dùng bfloat16.

## 1 · Thư viện CÓ SẴN (quan trọng: quyết định bạn phải `pip install` gì trong phòng thi)

In [ ]:
import importlib.metadata as md_
want = ["torch","torchvision","transformers","tokenizers","accelerate","timm","datasets",
        "scikit-learn","lightgbm","xgboost","catboost","opencv-python","albumentations",
        "sacrebleu","sentencepiece","pytorch-lightning","fastai","shapely","optuna","pandas","numpy"]
have, missing = {}, []
for p in want:
    try: have[p] = md_.version(p)
    except md_.PackageNotFoundError: missing.append(p)
print("=== ĐÃ CÀI SẴN ===")
for k,v in have.items(): print(f"  {k:22s} {v}")
print("\n=== THIẾU (phải pip install trong giờ thi) ===")
for m in missing: print("  ", m)

## 2 · Thời gian `pip install` — trong 6 tiếng, 90 giây × 3 lần thử = 5 phút mất trắng

In [ ]:
import time, subprocess
def timed_install(pkg):
    t0=time.time()
    r=subprocess.run([sys.executable,"-m","pip","install","-q",pkg],capture_output=True,text=True)
    return round(time.time()-t0,1), r.returncode
for pkg in ["sacrebleu","sentencepiece"]:
    dt, rc = timed_install(pkg)
    print(f"pip install {pkg:16s} {dt:6.1f}s  (rc={rc})")

## 3 · CV — `resnet34` @224px, 3.000 ảnh, AMP bật/tắt

In [ ]:
import torch, torch.nn as nn, time
from torch.utils.data import TensorDataset, DataLoader
import torchvision

def bench_cv(amp, bs=32, n=3000, res=224, model_name="resnet34"):
    dev="cuda"
    m=getattr(torchvision.models,model_name)(weights=None).to(dev)
    opt=torch.optim.AdamW(m.parameters(),1e-3)
    scaler=torch.amp.GradScaler("cuda",enabled=amp)
    lossf=nn.CrossEntropyLoss()
    X=torch.randn(n,3,res,res); y=torch.randint(0,10,(n,))
    dl=DataLoader(TensorDataset(X,y),batch_size=bs,shuffle=True,num_workers=2,pin_memory=True)
    torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); t0=time.time()
    for xb,yb in dl:
        xb,yb=xb.to(dev,non_blocking=True),yb.to(dev,non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda",enabled=amp):
            loss=lossf(m(xb),yb)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    torch.cuda.synchronize()
    return round(time.time()-t0,1), round(torch.cuda.max_memory_allocated()/2**20)

for amp in (False, True):
    sec,mem = bench_cv(amp)
    print(f"resnet34 224px bs32 AMP={str(amp):5s} -> {sec:6.1f} s/epoch | {mem:5d} MB")
    print(f"   -> 20 phút inference đủ cho ~{int(3000*1200/sec):,} ảnh (ước lượng thô, train chậm hơn infer ~3x)")

## 4 · Batch size tối đa trước khi OOM (bạn quen 80GB, giờ chỉ 16GB)

In [ ]:
def max_batch(model_name="resnet34", res=224):
    import torchvision, gc
    dev="cuda"; bs=16; last=0
    while bs<=1024:
        try:
            torch.cuda.empty_cache(); gc.collect()
            m=getattr(torchvision.models,model_name)(weights=None).to(dev)
            opt=torch.optim.AdamW(m.parameters(),1e-3)
            x=torch.randn(bs,3,res,res,device=dev); y=torch.randint(0,10,(bs,),device=dev)
            with torch.amp.autocast("cuda"):
                loss=nn.CrossEntropyLoss()(m(x),y)
            loss.backward(); opt.step()
            last=bs; bs*=2
            del m,opt,x,y,loss
        except torch.cuda.OutOfMemoryError:
            break
    torch.cuda.empty_cache(); gc.collect()
    return last
print("resnet34 @224 AMP — batch tối đa:", max_batch())

## 5 · NLP — fine-tune encoder ~100M tham số, 48.000 câu, max_len=96\n\n*(đúng quy mô bài R-ViHSD vòng trường: 48.092 dòng)*

In [ ]:
try:
    from transformers import AutoModelForSequenceClassification, AutoConfig
    name="xlm-roberta-base"          # ~278M; đổi 'distilbert-base-multilingual-cased' (~135M) nếu quá chậm
    cfg=AutoConfig.from_pretrained(name, num_labels=3)
    m=AutoModelForSequenceClassification.from_config(cfg).cuda()
    print("params:", round(sum(p.numel() for p in m.parameters())/1e6,1), "M")
    N, L, BS = 48092, 96, 32
    ids=torch.randint(0,1000,(N,L)); lab=torch.randint(0,3,(N,))
    dl=DataLoader(TensorDataset(ids,lab),batch_size=BS,shuffle=True,num_workers=2)
    opt=torch.optim.AdamW(m.parameters(),2e-5); scaler=torch.amp.GradScaler("cuda")
    torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); t0=time.time()
    STEPS=100                       # đo 100 bước rồi ngoại suy — chạy hết 48k quá lâu
    for i,(xb,yb) in enumerate(dl):
        if i>=STEPS: break
        xb,yb=xb.cuda(),yb.cuda(); opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            out=m(input_ids=xb,labels=yb)
        scaler.scale(out.loss).backward(); scaler.step(opt); scaler.update()
    torch.cuda.synchronize(); dt=time.time()-t0
    per_epoch=dt/STEPS*len(dl)
    print(f"{STEPS} bước: {dt:.1f}s -> 1 epoch ({len(dl)} bước) ≈ {per_epoch/60:.1f} phút")
    print(f"5 fold × 3 epoch ≈ {per_epoch*15/60:.0f} phút  <-- SO VỚI NGÂN SÁCH 6 TIẾNG!")
    print("peak VRAM:", round(torch.cuda.max_memory_allocated()/2**20), "MB")
except Exception as e:
    print("bỏ qua (thiếu transformers?):", e)

> **Đây là con số quan trọng nhất của cả notebook.**
> Ở vòng trường bạn chạy 5-fold trên H100. Nếu 5 fold × 3 epoch mất > 90 phút trên T4,
> thì trong phòng thi bạn **chỉ được 1 fold**, và mọi ước lượng OOF phải thiết kế lại.

## 6 · Sinh `env_report.md`

In [ ]:
rows=[]
rows.append(f"| Nền tảng | {'Kaggle' if os.path.exists('/kaggle') else 'Colab'} |")
rows.append(f"| GPU | {torch.cuda.get_device_name(0)} |")
rows.append(f"| VRAM | {round(torch.cuda.get_device_properties(0).total_memory/2**30,1)} GB |")
rows.append(f"| bf16 hỗ trợ | {torch.cuda.is_bf16_supported()} |")
rows.append(f"| CPU cores | {os.cpu_count()} |")
rows.append(f"| RAM | {round(os.sysconf('SC_PAGE_SIZE')*os.sysconf('SC_PHYS_PAGES')/2**30,1)} GB |")
rows.append(f"| torch | {torch.__version__} |")
print("# env_report.md — dán vào tuan01/env_report.md\n")
print("| Hạng mục | Giá trị |\n|---|---|")
print("\n".join(rows))
print("""
| resnet34 224px bs32 AMP=off | ___ s/epoch, ___ MB |
| resnet34 224px bs32 AMP=on  | ___ s/epoch, ___ MB |
| resnet34 batch tối đa @224  | ___ |
| encoder 100M, 48k câu, len96 | ___ phút/epoch |
| 5 fold × 3 epoch | ___ phút |
| pip install sacrebleu | ___ s |
| Thư viện THIẾU | ___ |

## Kết luận rút ra
- Trong 6 tiếng, tôi chạy được tối đa ___ epoch / ___ fold.
- Ngân sách 20 phút của `Final/main.py` đủ cho inference trên ___ mẫu.
- Ba thứ phải `pip install` đầu giờ thi: ___
""")

## 7 · Việc phải làm bằng tay (không tự động được)

- [ ] **Kaggle:** *Settings → Accelerator* xem còn bao nhiêu **giờ GPU trong tuần** (quota 30h). Ghi lại.
- [ ] **Colab:** để notebook chạy 30 phút rồi thử đóng tab 5 phút — xem có bị ngắt runtime không.
- [ ] Thử **mở 2 phiên Colab bằng 2 tài khoản Google trên cùng máy** (profile trình duyệt khác nhau)
      → xác nhận mẹo §7.4 trong kế hoạch: *2 máy ≠ 2 GPU*.
- [ ] Đo thời gian **lưu và tải lại checkpoint 500MB** lên Google Drive (chống mất tiến độ khi ngắt kết nối).